## Libraries

In [60]:
import pandas as pd
import numpy as np
import requests

## Loading Data

Retrieving Dataset from EIA api

In [61]:
'''
API_KEY = "rGIsttjuD3E6wa6wTqZwsgegfhWfRbE6CEMb6NiR"
url = "https://api.eia.gov/v2/electricity/rto/daily-region-sub-ba-data/data/"

params = {
    "api_key": API_KEY,
    "frequency": "daily",
    "data[0]": "value",
    "facets[subba][]": "PS",
    "start": "2022-04-01",
    "end": "2024-12-31",
    "length": 5000
}

r = requests.get(url, params=params)
data = r.json()

if "response" not in data or "data" not in data["response"]:
    raise ValueError("API returned no data")

df = pd.DataFrame(data["response"]["data"])

df.to_csv("../data/electric.csv", index=False)
'''

'\nAPI_KEY = "rGIsttjuD3E6wa6wTqZwsgegfhWfRbE6CEMb6NiR"\nurl = "https://api.eia.gov/v2/electricity/rto/daily-region-sub-ba-data/data/"\n\nparams = {\n    "api_key": API_KEY,\n    "frequency": "daily",\n    "data[0]": "value",\n    "facets[subba][]": "PS",\n    "start": "2022-04-01",\n    "end": "2024-12-31",\n    "length": 5000\n}\n\nr = requests.get(url, params=params)\ndata = r.json()\n\nif "response" not in data or "data" not in data["response"]:\n    raise ValueError("API returned no data")\n\ndf = pd.DataFrame(data["response"]["data"])\n\ndf.to_csv("../data/electric.csv", index=False)\n'

In [62]:
df_electric = pd.read_csv("../data/electric.csv")
print(df_electric.shape)
display(df_electric.head(3))

(5000, 8)


,period,subba,subba-name,parent,parent-name,timezone,value,value-units
0,2024-12-31,PS,Public Service Electric & Gas zone,PJM,"PJM Interconnection, LLC",Arizona,105040,megawatthours
1,2024-12-31,PS,Public Service Electric & Gas zone,PJM,"PJM Interconnection, LLC",Central,105088,megawatthours
2,2024-12-31,PS,Public Service Electric & Gas zone,PJM,"PJM Interconnection, LLC",Eastern,105170,megawatthours


In [63]:
df_noaa = pd.read_csv("../data/noaa.csv", low_memory=False)
print(df_noaa.shape)
display(df_noaa.head(3))

(318144, 8)


,STATION,NAME,DATE,AWND,PRCP,TAVG,TMAX,TMIN
0,US1NJHN0019,"LEBANON 3.2 SW, NJ US",2022-01-01,NaN,0.05,NaN,NaN,NaN
1,US1NJHN0019,"LEBANON 3.2 SW, NJ US",2022-01-02,NaN,0.62,NaN,NaN,NaN
2,US1NJHN0019,"LEBANON 3.2 SW, NJ US",2022-01-03,NaN,0.00,NaN,NaN,NaN


## Data Cleaning

### Df_electric

In [64]:
# Rename period column to date

df_electric["DATE"] = pd.to_datetime(df_electric["period"])  
df_electric["DATE"] = df_electric["DATE"].dt.date  

In [65]:
# Rename value to demand

df_electric = df_electric.rename(columns={'value': 'demand'})  

In [66]:
full_range = pd.date_range(start="2022-04-01", end="2024-12-31", freq="D").date
missing_dates = sorted(set(full_range) - set(df_electric["DATE"]))

print(f"Missing dates: {len(missing_dates)}")
print(missing_dates)

Missing dates: 3
[datetime.date(2024, 3, 10), datetime.date(2024, 11, 2), datetime.date(2024, 11, 3)]


In [67]:
missing_df = pd.DataFrame({"DATE": missing_dates})
df_electric = pd.concat([df_electric, missing_df], ignore_index=True)
df_electric = df_electric.sort_values("DATE").reset_index(drop=True)

In [68]:
df_electric.dropna(inplace=True)

In [69]:
print(df_electric.isna().sum())

period         0
subba          0
subba-name     0
parent         0
parent-name    0
timezone       0
demand         0
value-units    0
DATE           0
dtype: int64


### Df_noaa

In [70]:
# Convert DATE column to datetime
df_noaa["DATE"] = pd.to_datetime(df_noaa["DATE"])

# Need only daily
df_noaa["DATE"] = df_noaa["DATE"].dt.date

In [71]:
# Check for missing dates / duplicates

full_range = pd.date_range(start="2022-04-01", end="2024-12-31", freq="D")

missing_dates = full_range.difference(df_noaa["DATE"])

print("Number of missing rows:", len(missing_dates))
print(missing_dates)

# check for duplicates
duplicates = df_noaa[df_noaa.duplicated(subset=["DATE"], keep=False)]
print("Duplicate rows:", len(duplicates))

Number of missing rows: 0
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Duplicate rows: 318144


In [72]:
# Function to take the first non-NaN value
def first_non_na(series):
    non_na_values = series.dropna()
    if not non_na_values.empty:
        return non_na_values.iloc[0]
    else:
        return np.nan

df_noaa = df_noaa.groupby('DATE', as_index=False).agg(first_non_na)

In [73]:
print(df_noaa.isna().sum())

DATE       0
STATION    0
NAME       0
AWND       0
PRCP       0
TAVG       0
TMAX       0
TMIN       0
dtype: int64


### Data Merging

In [74]:
df_noaa["DATE"] = pd.to_datetime(df_noaa["DATE"]).dt.date  
df_electric["DATE"] = pd.to_datetime(df_electric["DATE"]).dt.date  

df_merged = pd.merge(df_electric, df_noaa, on="DATE", how="left")

In [75]:
df_merged.to_csv("../data/pjm_noaa_merged.csv", index=False)